See here: 
https://viewer.mathworks.com/?viewer=plain_code&url=https%3A%2F%2Fwww.mathworks.com%2Fmatlabcentral%2Fmlc-downloads%2Fdownloads%2Fdb12d59b-8cd9-448f-a2dd-799d73a27d50%2F80ce221e-66ee-44cd-8da6-5365ac1b8f60%2Ffiles%2FpGKB_Tikhonov%2FIRtools-master%2FExtra%2FFlexGK.m&embed=web

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [9]:


def flex_gk(A, U, M, T, V, L):
    """
    Perform one step of (inexact/flexible) Golub Kahan with reorthogonalization.

    Parameters:
        A : tuple
            Functions for matrix A and A-transpose operations.
        U : ndarray
            Accumulation of orthonormal vectors (k+1) x k matrix.
        M : ndarray
            k x k upper triangular matrix.
        T : ndarray
            (k+1) x k upper Hessenberg matrix.
        V : ndarray
            Accumulation of orthonormal vectors.
        L : ndarray
            Changing preconditioner matrix.
        A_times_vec : function
            Function to perform matrix-vector multiplication with A.
        Atransp_times_vec : function
            Function to perform matrix-vector multiplication with A-transpose.

    Returns:
        U : ndarray
            Updated U matrix with orthonormal columns.
        M : ndarray
            Updated M matrix.
        T : ndarray
            Updated T matrix.
        V : ndarray
            Updated V matrix.
        z : ndarray
            New z vector.
    """
    # Get the previous dimension
    k = T.shape[1] + 1

    # Step 1: Update v
    v = (A.T @ U[:,k-1]).reshape(-1)
    # v = Atransp_times_vec(A, U[:, k - 1]).reshape(-1)
    
    if k > 1:
        for j in range(k - 1):
            M[j, k - 1] = np.dot(V[:, j], v)
            v = v - M[j, k - 1] * V[:, j]
    
    M[k - 1, k - 1] = np.linalg.norm(v)
    v = v / M[k - 1, k - 1]
    
    # Step 2: Compute new z
    z = np.dot(L, v)
    
    # Step 3: Update u
    # u = A_times_vec(A, z).reshape(-1)
    u = (A @ z).reshape(-1)
    
    for j in range(k):
        T[j, k - 1] = np.dot(U[:, j], u)
        u = u - T[j, k - 1] * U[:, j]
    
    T[k, k - 1] = np.linalg.norm(u)
    u = u / T[k, k - 1]
    
    # Step 4: Update U and V matrices
    U = np.column_stack((U, u))
    V = np.column_stack((V, v))

    return U, M, T, V, z


In [7]:
m = 20
n = 10
k = 18

A = np.random.normal((m,n))
L = np.random.normal((k,n))
b = np.random.normal(size=m)

U = b/np.linalg.norm(b)
U = U[:,None]

In [10]:
flex_gk(A, U, None, None, None, L)

AttributeError: 'NoneType' object has no attribute 'shape'